In [2]:
!pip install rapidfuzz spacy
!python -m spacy download en_core_web_sm

import os
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz

# 1. Create the provided text files in the Colab environment
with open("fatma.txt", "w", encoding="utf-8") as f:
    f.write("Life is all about doing your best in trying to\nfind what works out for you and taking most time in\ntrying to pursue those skills")
with open("john.txt", "w", encoding="utf-8") as f:
    f.write("Life is all about finding money and spending on luxury stuffs\nCoz this life is kinda short , trust")
with open("juma.txt", "w", encoding="utf-8") as f:
    f.write("Life to me is about finding money and use it on things that makes you happy\ncoz this life is kinda short")

# 2. Load Documents
student_files = [doc for doc in os.listdir() if doc.endswith('.txt')]
student_notes = [open(_file, encoding='utf-8').read() for _file in student_files]

# 3. Text Normalization (spaCy)
print("Loading spaCy model and normalizing text...")
nlp = spacy.load("en_core_web_sm")

def normalize_text(text):
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and not token.is_space]
    return " ".join(tokens)

normalized_notes = [normalize_text(text) for text in student_notes]

# 4. N-gram Features & TF-IDF Vectorization
def vectorize(texts):
    # Added ngram_range=(1,2) to fulfill dashboard requirements
    return TfidfVectorizer(ngram_range=(1, 2)).fit_transform(texts).toarray()

vectors = vectorize(normalized_notes)
s_vectors = list(zip(student_files, vectors, student_notes))

# 5. Cosine Similarity, RapidFuzz, and Thresholding Report
COSINE_THRESHOLD = 0.15
FUZZY_THRESHOLD = 40.0

def check_plagiarism():
    plagiarism_results = set()
    global s_vectors

    print("\n--- Plagiarism Detection Report ---")
    for student_a, text_vector_a, raw_text_a in s_vectors:
        new_vectors = s_vectors.copy()
        current_index = new_vectors.index((student_a, text_vector_a, raw_text_a))
        del new_vectors[current_index]

        for student_b, text_vector_b, raw_text_b in new_vectors:
            # Cosine Similarity
            sim_score = cosine_similarity([text_vector_a, text_vector_b])[0][1]

            # RapidFuzz Fuzzy Matching
            fuzzy_score = fuzz.token_sort_ratio(raw_text_a, raw_text_b)

            # Check against thresholds
            if sim_score > COSINE_THRESHOLD or fuzzy_score > FUZZY_THRESHOLD:
                student_pair = sorted((student_a, student_b))
                score_tuple = (student_pair[0], student_pair[1], sim_score, fuzzy_score)
                plagiarism_results.add(score_tuple)

    return plagiarism_results

# 6. Generate Final Output
results = check_plagiarism()
for data in sorted(results):
    print(f"\n[!] Potential Plagiarism Flagged between {data[0]} and {data[1]}")
    print(f"    Cosine Similarity: {data[2]:.4f} (Threshold: {COSINE_THRESHOLD})")
    print(f"    Fuzzy Match Score: {data[3]:.2f}% (Threshold: {FUZZY_THRESHOLD}%)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 27.9 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Loading spaCy model and normalizing text...

--- Plagiarism Detection Report ---

[!] Potential Plagiarism Flagged between fatma.txt and john.txt
    Cosine Similarity: 0.0590 (Threshold: 0.15)
    Fuzzy Match Score: 48.67% (Threshold: 40.0%)

[!] Potential Plagiarism Flagged between fatma.txt and juma.txt
    Cosine Similarity: 0.0590 (Threshold: 0.15)
    Fuzzy Match Score: 55.17% (Threshold: 40.0%)

[!] Potential Plagiarism Flagged between john.txt and juma.txt
    Cosine Similarity: 0.4357 (Threshold: 0.15)
    Fuzzy Match Score: 68.32% (Threshold: 40.0%)
